# CS5785 Assignment 1 - Programming Exercises, Part II
## Titanic

In this notebook, I work through the three questions in Part I using the Kaggle Titanic dataset. I put the explanation, code, and output for each question in its own section.

## 1. Load the Kaggle data

I downloaded the data from Kaggle. I load the training data from `titanic_train.csv`, the test data from `titanic_test.csv`, and the submission template from `gender_submission.csv`.

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 30)
DATA_DIR = Path('.')

train = pd.read_csv(DATA_DIR / 'titanic_train.csv')
test = pd.read_csv(DATA_DIR / 'titanic_test.csv')
submission_template = pd.read_csv(DATA_DIR / 'gender_submission.csv')

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
missing_summary = pd.DataFrame({
    'train_missing': train.isna().sum(),
    'test_missing': test.isna().sum()
}).fillna(0).astype(int)
missing_summary = missing_summary[missing_summary.sum(axis=1) > 0]

display(missing_summary)

,train_missing,test_missing
Age,177,86
Cabin,687,327
Embarked,2,0
Fare,0,1


`Cabin` and `Age` have the most missing values. I handle missing data in the preprocessing pipeline instead of deleting rows.

### Feature engineering and preprocessing

I use these features:

- `Pclass` and `Fare` represent socioeconomic and travel-class differences that may have affected access to lifeboats.
- `Sex` and `Age` capture the historical evacuation priority given to women and children.
- `SibSp` and `Parch` describe family relationships aboard the ship, while `Embarked` can capture differences among passenger groups and ticket classes.
- `FamilySize` and `IsAlone` summarize whether a passenger traveled alone or with relatives, which may have influenced evacuation behavior.
- `Title` is extracted from `Name` and grouped into common titles plus `Rare`; it provides information related to age, sex, and social status without using the high-cardinality raw name.
- `CabinKnown` records whether a passenger has cabin information, which may indirectly reflect passenger class and location on the ship.

I leave out `PassengerId`, `Name`, `Ticket`, and the raw `Cabin` column because they are identifiers, have many unique values, or contain too much missing data.

In [3]:
def engineer_features(frame):
    result = frame.copy()
    result['FamilySize'] = result['SibSp'] + result['Parch'] + 1
    result['IsAlone'] = (result['FamilySize'] == 1).astype(int)
    result['Title'] = result['Name'].str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
    common_titles = {'Mr', 'Mrs', 'Miss', 'Master'}
    result['Title'] = result['Title'].where(result['Title'].isin(common_titles), 'Rare')
    result['CabinKnown'] = result['Cabin'].notna().astype(int)
    return result

train_features = engineer_features(train)
test_features = engineer_features(test)

FEATURES = [
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
    'FamilySize', 'IsAlone', 'Title', 'CabinKnown'
]
NUMERIC_FEATURES = [
    'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'CabinKnown'
]
CATEGORICAL_FEATURES = ['Pclass', 'Sex', 'Embarked', 'Title']

X = train_features[FEATURES]
y = train_features['Survived'].astype(int)
X_test = test_features[FEATURES]

display(train_features[['Name', 'Title', 'FamilySize', 'IsAlone', 'CabinKnown']].head())

,Name,Title,FamilySize,IsAlone,CabinKnown
0,"Braund, Mr. Owen Harris",Mr,2,0,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0,1
2,"Heikkinen, Miss. Laina",Miss,1,1,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0,1
4,"Allen, Mr. William Henry",Mr,1,1,0


## 2. Data Preprocessing and Logistic Regression
We used a scikit-learn pipeline for data preprocessing.

We handled numerical and categorical features separately:

- **Numerical features:** We filled missing values with the median and standardized them.
- **Categorical features:** We filled missing values with the most frequent value and used one-hot encoding.

Finally, we combined these preprocessing steps with logistic regression in a pipeline.

In [4]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES)
])

model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

## 3. Training

I fit the pipeline using all 891 training rows, predict `Survived` for the 418 test passengers, and save the predictions in Kaggle's required format.

In [5]:
model.fit(X, y)
test_predictions = model.predict(X_test).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'].astype(int),
    'Survived': test_predictions
})
submission_path = DATA_DIR / 'titanic_submission.csv'
submission.to_csv(submission_path, index=False)

assert list(submission.columns) == ['PassengerId', 'Survived']
assert len(submission) == len(test) == len(submission_template)
assert submission['PassengerId'].equals(submission_template['PassengerId'])
assert set(submission['Survived'].unique()).issubset({0, 1})
assert not submission.isna().any().any()

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
